# Fine-tune PP-OCRv6 on your own data

This notebook fine-tunes PaddleOCR's PP-OCRv6 text detector and recognizer on the CORD receipt dataset, then measures both models against the stock ones on the same test set.

Before you start: **Runtime → Change runtime type → T4 GPU**. A full run takes about 1.5 hours on a free T4.

To train on your own images, label them with [PPOCRLabel](https://github.com/PFCCLab/PPOCRLabel) and follow step 2b.

## 1. Set up

In [ ]:
!git clone -q https://github.com/Dmitry543/Fine-Tuning-OCR-Model-with-PaddleOCR.git
%cd Fine-Tuning-OCR-Model-with-PaddleOCR
!bash setup.sh --gpu

## 2a. Prepare CORD

Downloads CORD-v2 (CC BY 4.0, 2.3 GB) from Hugging Face and converts it to PaddleOCR labels: 800 train, 100 validation and 100 test receipts, about 16,000 text lines.

In [ ]:
!python tools/prepare_data.py cord --out data/cord

## 2b. Or prepare your own data

Upload the folder PPOCRLabel saved (it contains `Label.txt`), then run the cell below instead of 2a and set `DATA = "data/mydata"` in the following cells.

In [ ]:
# !python tools/prepare_data.py ppocrlabel --input /content/my_images --out data/mydata

In [ ]:
DATA = "data/cord"

## 3. Measure the stock models

Detection F1, recognition accuracy on ground-truth line crops, and end-to-end F1 (box and exact text both right).

In [ ]:
import os
os.environ["PYTHONPATH"] = "third_party/PaddleOCR"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
RESULTS = f"results/{os.path.basename(DATA)}"
!python tools/evaluate.py --name stock --data {DATA} --out-dir {RESULTS} --device gpu:0 --det PP-OCRv6_small_det --rec PP-OCRv6_small_rec

## 4. Fine-tune the detector

The model is evaluated on the validation split during training and the best checkpoint is kept in `output/det/best_accuracy`. Edit `configs/det_ppocrv6_small.yml` to change epochs or learning rate.

In [ ]:
!python third_party/PaddleOCR/tools/train.py -c configs/det_ppocrv6_small.yml -o \
    Train.dataset.data_dir={DATA}/ "Train.dataset.label_file_list=[{DATA}/det_train.txt]" \
    Eval.dataset.data_dir={DATA}/ "Eval.dataset.label_file_list=[{DATA}/det_val.txt]"

## 5. Fine-tune the recognizer

In [ ]:
!python third_party/PaddleOCR/tools/train.py -c configs/rec_ppocrv6_small.yml -o \
    Train.dataset.data_dir={DATA}/ "Train.dataset.label_file_list=[{DATA}/rec_train.txt]" \
    Eval.dataset.data_dir={DATA}/ "Eval.dataset.label_file_list=[{DATA}/rec_val.txt]"

## 6. Export for inference

Exported models load directly into `paddleocr.PaddleOCR` (see step 9).

In [ ]:
for task in ["det", "rec"]:
    !python third_party/PaddleOCR/tools/export_model.py -c configs/{task}_ppocrv6_small.yml -o \
        Global.pretrained_model=output/{task}/best_accuracy Global.save_inference_dir=output/inference/{task}

## 7. Compare

In [ ]:
!python tools/evaluate.py --name finetuned --data {DATA} --out-dir {RESULTS} --device gpu:0 --det output/inference/det --rec output/inference/rec
!python tools/evaluate.py --compare {RESULTS}/stock.json {RESULTS}/finetuned.json

## 8. See the difference

In [ ]:
!python tools/visualize.py {RESULTS}/stock_predictions.json {RESULTS}/finetuned_predictions.json --data {DATA} --out comparison.jpg --images 2
from IPython.display import Image
Image("comparison.jpg")

## 9. Use the fine-tuned models

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv6_small_det", text_detection_model_dir="output/inference/det",
    text_recognition_model_name="PP-OCRv6_small_rec", text_recognition_model_dir="output/inference/rec",
    use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
)
for res in ocr.predict(f"{DATA}/images/test/test_0000.jpg"):
    print(res["rec_texts"])

## 10. Keep your models

Colab deletes files when the session ends. Download `models.zip` or copy it to Google Drive.

In [ ]:
!cd output && zip -qr ../models.zip inference
# from google.colab import drive; drive.mount("/content/drive"); !cp models.zip /content/drive/MyDrive/